Imports

In [15]:
import pandas as pd
import re

In [16]:
oscars = pd.read_csv("the_oscar_award.csv")



In [17]:
oscars_film = (
    oscars
    .groupby(["film", "year_film"])
    .agg(
        oscar_nominations=("category","count"),
        oscar_wins = ("winner", "sum")
    )
    .reset_index()
)

In [18]:
oscars_film["oscar_nominated"] = oscars_film["oscar_nominations"] > 0
oscars_film["oscar_won"] = oscars_film["oscar_wins"] > 0

In [19]:
def clean_title(title):

    title = str(title).lower()
    title = re.sub(r"\(.*?\)", "", title)   # remove brackets
    title = re.sub(r"[^a-z0-9 ]", "", title)
    title = title.strip()

    return title

In [20]:
df_full = pd.read_csv("movies_dataset_trimmed.csv")
df_full["title_clean"] = df_full["title"].apply(clean_title)
oscars_film["title_clean"] = oscars_film["film"].apply(clean_title)

In [21]:
df_full["release_year"] = pd.to_datetime(df_full["release_date"]).dt.year

In [22]:
df_merged = df_full.merge(
    oscars_film,
    left_on=["title_clean", "release_year"],
    right_on=["title_clean", "year_film"],
    how="left"
)

In [23]:
df_merged["oscar_nominations"] = df_merged["oscar_nominations"].fillna(0)
df_merged["oscar_wins"] = df_merged["oscar_wins"].fillna(0)

df_merged["oscar_nominated"] = df_merged["oscar_nominated"].fillna(False)
df_merged["oscar_won"] = df_merged["oscar_won"].fillna(False)

In [24]:
df_merged["oscar_nominated"].mean()

np.float64(0.14763948497854076)

In [25]:
df_merged

,title,release_date,budget,revenue,runtime,genres,release_year,title_clean,film,year_film,oscar_nominations,oscar_wins,oscar_nominated,oscar_won
0,GoodFellas,1990-09-12,25000000.0,47072327.0,145.0,"['Drama', 'Crime']",1990,goodfellas,NaN,NaN,0.0,0.0,False,False
1,Ghost,1990-07-13,22000000.0,505000000.0,127.0,"['Fantasy', 'Thriller', 'Romance']",1990,ghost,Ghost,1990.0,5.0,2.0,True,True
2,Total Recall,1990-06-01,65000000.0,261317921.0,113.0,"['Action', 'Adventure', 'Science Fiction']",1990,total recall,Total Recall,1990.0,3.0,1.0,True,True
3,Edward Scissorhands,1990-12-07,20000000.0,86024005.0,105.0,"['Fantasy', 'Drama', 'Romance']",1990,edward scissorhands,Edward Scissorhands,1990.0,1.0,0.0,True,False
4,Parallel College,1990-12-31,0.0,0.0,0.0,"['Comedy', 'Drama']",1990,parallel college,NaN,NaN,0.0,0.0,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3490,The Roundup 4: Punishment,2024-04-24,0.0,83703350.0,109.0,"['Action', 'Crime', 'Thriller', 'Comedy']",2024,the roundup 4 punishment,NaN,NaN,0.0,0.0,False,False
3491,Carry-On,2024-12-05,47000000.0,0.0,120.0,"['Thriller', 'Action']",2024,carryon,NaN,NaN,0.0,0.0,False,False
3492,Call Festival - Festspielauftakt 2024,2024-07-27,0.0,0.0,0.0,['Music'],2024,call festival festspielauftakt 2024,NaN,NaN,0.0,0.0,False,False
3493,Året der gik- 2024,2024-12-25,0.0,0.0,0.0,"['Family', 'Music', 'Documentary']",2024,ret der gik 2024,NaN,NaN,0.0,0.0,False,False


In [28]:
df_merged.to_csv("oscars_movies_merged.csv", index=False)
